# 03 · Tamlish zero-shot test (Colab GPU)

Does any existing open model already handle Tamil-English code-mixed speech? We test five candidates on:

- **own_ta**: 30 Tamlish sentences recorded by a native speaker (10 light, 10 medium, 10 heavy mixing)
- **indicvoices_ta**: 30 random Tamil clips from AI4Bharat IndicVoices (real, mostly spontaneous speech)

Metrics (see `src/tamlish.py`): strict WER, mixed error rate (MER), script-agnostic phonetic CER, English-word
recall (in either script), and wrong-language rate. Runtime: T4 GPU, `HF_TOKEN` secret set.

## 1 · Setup (run after every restart)

In [ ]:
import os, sys, subprocess, importlib
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
ROOT = "/content/drive/MyDrive/local-voice-research"
DATA, RESULTS = f"{ROOT}/data", f"{ROOT}/results_tamlish"
os.makedirs(RESULTS, exist_ok=True)
if os.path.isdir("/content/lvr"):
    subprocess.run(["git", "-C", "/content/lvr", "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/Kiranrossi/local-voice-research", "/content/lvr"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "whisper_normalizer"])
sys.path.insert(0, "/content/lvr/src")
import harness as H, tamlish as TL
importlib.reload(H); importlib.reload(TL)
import gc, numpy as np, torch
import transformers; transformers.logging.set_verbosity_error()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

SETS = ("own_ta", "indicvoices_ta")
def run_tamlish(model, name, variant="", **kw):
    for ts in SETS:
        m = f"{DATA}/{ts}/manifest.jsonl"
        if os.path.exists(m):
            H.evaluate(model, name, m, ts, RESULTS, variant=variant, normaliser=TL.normalise,
                       objective="obj2", notes="tamlish strict normaliser", **kw)
        else:
            print("skip", ts, "(not prepared)")

def hf_pipe(model_id, **kw):
    from transformers import pipeline
    try:
        return pipeline("automatic-speech-recognition", model=model_id, dtype=torch.float16, device="cuda:0", **kw)
    except TypeError:
        return pipeline("automatic-speech-recognition", model=model_id, torch_dtype=torch.float16, device="cuda:0", **kw)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 2 · Prepare the two Tamil test sets (once, about 3 minutes)

In [ ]:
import io, soundfile as sf, librosa
from datasets import load_dataset, Audio

# own_ta: the 30 Tamlish recordings (normally created by notebook 01)
if not os.path.exists(f"{DATA}/own_ta/manifest.jsonl"):
    clips = H.read_manifest(f"{DATA}/recordings/manifest.jsonl")
    H.write_manifest([c for c in clips if c["id"].startswith("ta_")], f"{DATA}/own_ta/manifest.jsonl")
print("own_ta:", len(H.read_manifest(f"{DATA}/own_ta/manifest.jsonl")), "clips")

# indicvoices_ta: 30 random Tamil clips (2 to 20 s) from the validation split, streamed
out = f"{DATA}/indicvoices_ta"
if not os.path.exists(f"{out}/manifest.jsonl"):
    os.makedirs(out, exist_ok=True)
    ds = load_dataset("ai4bharat/IndicVoices", "tamil", split="valid", streaming=True)
    cols = list(ds.features.keys()) if ds.features else ["audio_filepath", "text"]
    audio_col = next(k for k in ("audio_filepath", "audio") if k in cols)
    text_col = next(k for k in ("text", "transcript", "verbatim", "normalized") if k in cols)
    print("columns:", cols, "| audio:", audio_col, "| text:", text_col)
    ds = ds.cast_column(audio_col, Audio(decode=False)).shuffle(seed=42, buffer_size=300)
    rows = []
    for ex in ds:
        a = ex[audio_col]
        raw = a["bytes"] if a.get("bytes") else open(a["path"], "rb").read()
        audio, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
        audio = audio.mean(axis=1)
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        dur = len(audio) / 16000
        if not (2.0 <= dur <= 20.0) or not ex[text_col]:
            continue
        cid = f"iv_ta_{len(rows):03d}"
        path = f"{out}/{cid}.wav"; sf.write(path, audio, 16000, subtype="PCM_16")
        meta = {k: ex[k] for k in ("gender", "age_group", "district", "state", "task_name", "scenario") if k in ex and isinstance(ex[k], str)}
        rows.append({"id": cid, "audio_filepath": path, "text": ex[text_col], "duration": round(dur, 3), **meta})
        if len(rows) >= 30:
            break
    H.write_manifest(rows, f"{out}/manifest.jsonl")
iv = H.read_manifest(f"{out}/manifest.jsonl")
print("indicvoices_ta:", len(iv), "clips; example:", iv[0]["text"][:80])
print("clips containing Latin-script words:", sum(TL.is_latin(t) for c in iv for t in TL.normalise(c["text"]).split()))

## 3 · Whisper large-v3-turbo (Tamil forced, then automatic language detection)

In [ ]:
class HFWhisper:
    def __init__(self, model_id, language=None): self.model_id, self.language = model_id, language
    def load(self):
        if not hasattr(self, "pipe"): self.pipe = hf_pipe(self.model_id)
    def transcribe(self, clip):
        gk = {"task": "transcribe"}
        if self.language: gk["language"] = self.language
        return self.pipe({"raw": clip["audio"].copy(), "sampling_rate": 16000}, generate_kwargs=gk)["text"]

model = HFWhisper("openai/whisper-large-v3-turbo", "ta")
run_tamlish(model, "whisper-large-v3-turbo", variant="language=ta")
model.language = None
run_tamlish(model, "whisper-large-v3-turbo", variant="language=auto")
del model; free_gpu()

## 4 · Whisper large-v3 (full decoder, Tamil forced)

In [ ]:
model = HFWhisper("openai/whisper-large-v3", "ta")
run_tamlish(model, "whisper-large-v3", variant="language=ta")
del model; free_gpu()

## 5 · Whisper medium fine-tuned on Tamil (community model)

`vasista22/whisper-tamil-medium` (Whisper medium fine-tuned on public Tamil datasets; community model, evidence grade C).
If it fails to load, skip it.

In [ ]:
model = HFWhisper("vasista22/whisper-tamil-medium", "ta")
run_tamlish(model, "whisper-tamil-medium", variant="vasista22, language=ta")
del model; free_gpu()

## 6 · AI4Bharat IndicConformer 600M (Tamil RNNT head)

In [ ]:
!pip install -q onnxruntime-gpu

In [ ]:
from transformers import AutoModel

class IndicConformer:
    def __init__(self, decoder="rnnt"): self.decoder = decoder
    def load(self):
        if not hasattr(self, "m"):
            self.m = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)
    def transcribe(self, clip):
        wav = torch.from_numpy(clip["audio"].copy()).unsqueeze(0)
        out = self.m(wav, "ta", self.decoder)
        return out if isinstance(out, str) else (out[0] if isinstance(out, (list, tuple)) else str(out))

model = IndicConformer("rnnt")
run_tamlish(model, "indicconformer-600m", variant="ta, rnnt", runtime="onnx (custom code)")
del model; free_gpu()

## 7 · Score everything

Prints the main table, then a few side-by-side outputs per mixing level so you can sanity-check them.

In [ ]:
import pandas as pd
rows = TL.score_results(RESULTS)
df = pd.DataFrame(rows)
cols = ["model", "variant", "test_set", "strict_wer", "mer", "phon_cer", "en_recall", "wrong_language",
        "phon_cer_light", "phon_cer_medium", "phon_cer_heavy", "latency_median_s"]
pd.set_option("display.width", 250)
print(df[cols].to_string(index=False))

In [ ]:
# Side-by-side outputs on your own clips: one light, one medium, one heavy sentence
import json, glob
ids = ["ta_03", "ta_15", "ta_26"]
for cid in ids:
    print("=" * 100)
    for r in rows:
        if r["test_set"] != "own_ta": continue
        for line in open(f"{RESULTS}/outputs/{r['run_id']}.tamlish.jsonl", encoding="utf-8"):
            o = json.loads(line)
            if o["id"] == cid:
                print(f"[{cid}] {r['model']} ({r['variant']}):\n    REF: {o['ref']}\n    HYP: {o['hyp']}")

In [ ]:
!cd /content/drive/MyDrive/local-voice-research && zip -qr /content/tamlish_results.zip results_tamlish && echo zipped
from google.colab import files; files.download("/content/tamlish_results.zip")